In [1]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from plot_func import error_scatter, interval_score, coverage, performance_dist

folder = "interp_reg_spatial"

metrics = ["total_plus", "total_minus"]
models = [
    "1", "2", "3", "4", "5", "6", "7", "8"
]

TOLERANCE=0.1

In [2]:

for metric in metrics:
    
    dfs = []
    for model in models:
                df = pd.read_csv(f"../../{folder}/ablation_results/{metric}_ablation_{model}.csv")
                df["model"] = model
                dfs.append(df)

    df_all = pd.concat(dfs, ignore_index=True)
    
    fig, ax = error_scatter(df_all, TOLERANCE, tag=metric)
    save_path = f"../../{folder}/plots/error_scatter_{metric}_{model}.jpg"
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    fig, ax = interval_score(df_all, models, tag=metric)
    save_path = f"../../{folder}/plots/interval_score_{metric}_{model}.jpg"
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    fig, ax = coverage(df_all, tag=metric)
    save_path = f"../../{folder}/plots/coverage_{metric}_{model}.jpg"
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.
/home/wiera/Documents/fullfieldvalmetrics/scripts/plots/plot_func.py:101: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_all.groupby("model")["within_pi"]
Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.
/home/wiera/Documents/fullfieldvalmetrics/scripts/plots/plot_func.py:101: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_all.groupby("model")["within_pi"]


In [3]:
all_results = []

for metric in metrics:

    for model in models:

        # filepath = (f"../../{folder}/ablation_results/"
        #             f"{metric}_ablation_{model_type_print}.csv"
        # )
        filepath = f"../../{folder}/ablation_summary_{model}.csv"

        df = pd.read_csv(filepath)

        # Store the model information with each result
        df["metric"] = metric
        df["model_type"] = model


        all_results.append(df)


# Combine all summary files
results = pd.concat(all_results, ignore_index=True)
print(results)

              d_type        MAE       RMSE          MAPE  mean_abs_error  \
0           sim_plus   3.718747   4.150580  3.059497e+01        3.718747   
1          sim_minus   3.478734   3.945821  3.053842e+01        3.478734   
2    model_form_plus   5.312046   7.088330  3.532726e+03        5.312046   
3   model_form_minus  10.098803  13.485606  1.986592e+06       10.098803   
4         total_plus   5.799666   6.650584  4.656173e+01        5.799666   
..               ...        ...        ...           ...             ...   
91         sim_minus   1.677817   1.857824  1.550226e+01        1.677817   
92   model_form_plus   9.669221  11.154203  1.492115e+04        9.669221   
93  model_form_minus   8.292451  11.028638  9.110277e+05        8.292451   
94        total_plus  10.203972  12.092039  6.680873e+01       10.203972   
95       total_minus   8.187703  10.313261  7.315416e+01        8.187703   

    mean_rel_error  mean_pi_width  mean_pi_error  pi_coverage  \
0         0.305950    

In [4]:
best_mean_rel_error = (
    results.loc[
        results.groupby("d_type")["mean_rel_error"].idxmin()
    ]
)

print(best_mean_rel_error[
    ["d_type", "model_type", "mean_rel_error", "pi_coverage", "mean_interval_score"]
])

              d_type model_type  mean_rel_error  pi_coverage  \
21  model_form_minus          4     7947.080601     0.857143   
2    model_form_plus          1       35.327258     0.857143   
13         sim_minus          3        0.115832     0.857143   
12          sim_plus          3        0.085609     1.000000   
23       total_minus          4        0.330249     1.000000   
40        total_plus          7        0.296205     1.000000   

    mean_interval_score  
21            98.478698  
2            112.167242  
13            28.813270  
12            20.215633  
23            52.381521  
40            85.235011  


In [5]:
best_interval_score = (
    results.loc[
        results.groupby("d_type")["mean_interval_score"].idxmin()
    ]
)

print(best_interval_score[
    ["d_type", "model_type", "mean_rel_error", "pi_coverage", "mean_interval_score"]
])

              d_type model_type  mean_rel_error  pi_coverage  \
21  model_form_minus          4     7947.080601     0.857143   
8    model_form_plus          2       61.513699     1.000000   
19         sim_minus          4        0.250526     1.000000   
12          sim_plus          3        0.085609     1.000000   
23       total_minus          4        0.330249     1.000000   
22        total_plus          4        0.434652     1.000000   

    mean_interval_score  
21            98.478698  
8             84.426693  
19            28.699035  
12            20.215633  
23            52.381521  
22            61.675877  


In [6]:
target_coverage = 0.95

results["coverage_distance"] = (
    results["pi_coverage"] - target_coverage
).abs()

best_coverage = (
    results.loc[
        results.groupby("d_type")["coverage_distance"].idxmin()
    ]
)

print(best_coverage[
    ["d_type", "model_type", "mean_rel_error", "pi_coverage", "mean_interval_score"]
])


             d_type model_type  mean_rel_error  pi_coverage  \
9  model_form_minus          2    27201.726255     0.857143   
8   model_form_plus          2       61.513699     1.000000   
1         sim_minus          1        0.305384     1.000000   
0          sim_plus          1        0.305950     1.000000   
5       total_minus          1        0.772259     1.000000   
4        total_plus          1        0.465617     1.000000   

   mean_interval_score  
9           347.139152  
8            84.426693  
1           124.728253  
0           128.547649  
5           243.138763  
4           156.178405  


In [7]:
results["rel_error_rank"] = (
    results.groupby("d_type")["mean_rel_error"]
    .rank(method="min", ascending=True)
)

results["interval_score_rank"] = (
    results.groupby("d_type")["mean_interval_score"]
    .rank(method="min", ascending=True)
)

results["coverage_rank"] = (
    results.groupby("d_type")["coverage_distance"]
    .rank(method="min", ascending=True)
)

results["overall_rank"] = (
    results["rel_error_rank"]
    + results["interval_score_rank"]
    + results["coverage_rank"]
)

In [8]:
best_overall = (
    results.loc[
        results.groupby("d_type")["overall_rank"].idxmin()
    ]
)

print(best_overall[
    [
        "d_type",
        "model_type",
        "mean_rel_error",
        "pi_coverage",
        "mean_interval_score",
        "overall_rank",
    ]
])

              d_type model_type  mean_rel_error  pi_coverage  \
21  model_form_minus          4     7947.080601     0.857143   
8    model_form_plus          2       61.513699     1.000000   
19         sim_minus          4        0.250526     1.000000   
12          sim_plus          3        0.085609     1.000000   
23       total_minus          4        0.330249     1.000000   
22        total_plus          4        0.434652     1.000000   

    mean_interval_score  overall_rank  
21            98.478698           3.0  
8             84.426693           5.0  
19            28.699035           9.0  
12            20.215633           3.0  
23            52.381521           3.0  
22            61.675877           5.0  


In [9]:
metrics_to_plot = {
    "mean_rel_error",
    "pi_coverage",
    "mean_interval_score",
}

for model_type in results["model_type"].unique():

    kernel_results = results[
        results["model_type"] == model_type
    ].copy()

    fig, axes = performance_dist(kernel_results, metrics_to_plot, model_type)
    save_path = f"../../{folder}/plots/perform_dist_{model_type}.jpg"
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)